In [10]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import CIFAR10
import torch.optim as optim

In [11]:
from torchvision.transforms import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_set = CIFAR10(root='./data', train=True, download=False, transform=transform)
test_set = CIFAR10(root='./data', train=False, download=False, transform=transform)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [13]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())

In [14]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
    epoch_train_loss = running_loss / len(train_loader)

    print(f'Epoch {epoch+1} / {epochs} train loss: {epoch_train_loss}')

Epoch 1 / 10 train loss: 1.3936277671390787
Epoch 2 / 10 train loss: 0.9645526359224563
Epoch 3 / 10 train loss: 0.7826801211861394
Epoch 4 / 10 train loss: 0.6645251631431872
Epoch 5 / 10 train loss: 0.5581349695430082
Epoch 6 / 10 train loss: 0.4750309146540549
Epoch 7 / 10 train loss: 0.39643936949160397
Epoch 8 / 10 train loss: 0.32439369180470784
Epoch 9 / 10 train loss: 0.2629878108134812
Epoch 10 / 10 train loss: 0.20901417492143334


In [16]:
model.eval()
correct_vals = 0
total_vals = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct_vals += (predicted == labels).sum().item()
        total_vals += labels.size(0)

print(f'Accuracy : {correct_vals / total_vals * 100}')

Accuracy : 74.24
